In [1]:
import pandas as pd
import openpyxl
import yaml
import os
import csv

In [2]:
# --- Configuration ---
transformations_dir = "../data_processing/transformations"
pathway_csv = "trans_specif_Uganda.csv"
strategy_defs_csv = os.path.join(transformations_dir, "strategy_definitions.csv")
excel_path = "../ssp_modeling/scenario_mapping/ssp_uganda_transformation_cw_bau.xlsx"

# --- Step 1: Load transformation metadata from Excel ---
wb = openpyxl.load_workbook(excel_path, data_only=True)

ws_yaml = wb["yaml"]
rows_yaml = list(ws_yaml.iter_rows(min_row=1, values_only=True))
headers_yaml = rows_yaml[0]

transformations = {
    row[1]: {h: row[i] for i, h in enumerate(headers_yaml) if i != 1}
    for row in rows_yaml[1:]
}

ws_main = wb["main"]
for row in ws_main.iter_rows(min_row=2, values_only=True):
    code, desc = row[2], row[4]
    if code in transformations:
        transformations[code]["transformation_description"] = desc

print(f"Loaded {len(transformations)} transformations from Excel")

# --- Step 2: Case-insensitive index of all YAML files ---
yaml_files = [
    f for f in os.listdir(transformations_dir)
    if f.endswith(".yaml") and f.startswith("transformation_")
]
yaml_index = {f.lower(): f for f in yaml_files}

# --- Step 3: Read pathway definitions from trans_specif_Uganda.csv ---
# Use strategy_code as pathway key and transformation_specification as TX list.
# Fall back to strategy_definitions.csv when the specification column is invalid.
fallback_specs = {}
with open(strategy_defs_csv, encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    for row in reader:
        strategy_code = (row.get("strategy_code") or "").strip()
        tx_specs = (row.get("transformation_specification") or "").strip()
        if strategy_code:
            fallback_specs[strategy_code] = tx_specs

pathways = {}
pathway_labels = {}

with open(pathway_csv, encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    for row in reader:
        strategy_code = (row.get("strategy_code") or "").strip()
        strategy_name = (row.get("strategy") or "").strip()
        specs = (row.get("transformation_specification") or "").strip()

        if not strategy_code:
            print("WARNING: Row without strategy_code, skipping")
            continue

        if not specs.startswith("TX:"):
            fallback = fallback_specs.get(strategy_code, "")
            if fallback.startswith("TX:"):
                specs = fallback
                print(f"INFO: Loaded specs for '{strategy_code}' from strategy_definitions.csv")
            else:
                print(f"WARNING: No valid TX codes for '{strategy_code}', skipping")
                continue

        tx_codes = [tx.strip() for tx in specs.split("|") if tx.strip().startswith("TX:")]
        if not tx_codes:
            print(f"WARNING: Empty TX list for '{strategy_code}', skipping")
            continue

        pathways[strategy_code] = tx_codes
        pathway_labels[strategy_code] = strategy_name or strategy_code

# --- Step 4: Map TX codes → base codes per pathway ---
KNOWN_SUFFIXES = [
    "_STRATEGY_NDC2020", "_STRATEGY_NDC_25", "_STRATEGY_NDC_2",
    "_STRATEGY_NDC_3", "_STRATEGY_NDC", "_STRATEGY_BAU",
    "_STRATEGY_ASP", "_STRATEGY_NZ", "_NDC_25", "_NDC_2", "_NZ",
]

def strip_suffix(tx_code):
    for sfx in KNOWN_SUFFIXES:
        if tx_code.endswith(sfx):
            return tx_code[:-len(sfx)], sfx
    return tx_code, ""

def tx_to_yaml_key(tx_code):
    stem = tx_code.replace("TX:", "").replace(":", "_").lower()
    return f"transformation_{stem}.yaml"

pw_tx_map = {}
extra_base_codes = set()

for pk, tx_list in pathways.items():
    mapping = {}
    for tx in tx_list:
        base, _ = strip_suffix(tx)
        mapping[base] = tx
        if base not in transformations:
            extra_base_codes.add(base)
    pw_tx_map[pk] = mapping

# Add transformations that only appear in pathway-specific YAMLs (no Excel entry)
for base_code in extra_base_codes:
    subsector = base_code.split(":")[1]
    name = base_code.split(":")[-1].replace("_", " ").title()
    transformations[base_code] = {
        "subsector": subsector,
        "transformation_name": name,
        "transformation_yaml_name": tx_to_yaml_key(base_code).replace("transformation_", ""),
        "transformation_description": "",
    }
    print(f"  Added from pathways: {base_code} → {name}")

print(f"\nLoaded {len(pathways)} pathways:")
for pk in pathways:
    print(f"  {pk} ({pathway_labels[pk]}): {len(pathways[pk])} TX codes")


Loaded 61 transformations from Excel
  Added from pathways: TX:SCOE:SHIFT_FUEL_HEAT_STRATEGY_NDC_25_ELEC_CM → Shift Fuel Heat Strategy Ndc 25 Elec Cm
  Added from pathways: TX:LNDU:DEC_WETLAND_LOSS → Dec Wetland Loss
  Added from pathways: TX:LNDU:PLUR → Plur
  Added from pathways: TX:SCOE:SHIFT_FUEL_HEAT_STRATEGY_NDC_25_ELEC_RES → Shift Fuel Heat Strategy Ndc 25 Elec Res
  Added from pathways: TX:SCOE:INC_EFFICIENCY_HEAT → Inc Efficiency Heat
  Added from pathways: TX:FRST:INCREASE_SEQUESTRATION → Increase Sequestration
  Added from pathways: TX:LNDU:SET_WETLANDS_MINIMUM → Set Wetlands Minimum

Loaded 5 pathways:
  PFLO:NDC_25_SCOE_ELEC (NDC 2.0 with a 2025 start tuned manually and buildings electrified at 50%): 35 TX codes
  PFLO:NDC_25_UNCONDITIONAL (NDC 2.0 Unconditional based on Table 2-55 from NDC Technical Report (Zutari)): 6 TX codes
  PFLO:NDC_25_UNCONDITIONAL_CE (NDC 2.0 Unconditional based on cost effective measures from tornado): 9 TX codes
  PFLO:HBLE (HBLE): 59 TX codes
 

In [3]:
# --- Step 5: Extract magnitude/parameters from YAML files ---
SKIP_PARAMS = {"vec_implementation_ramp", "return_pathways", "return_prevalence_dict"}

def extract_parameters(yaml_path):
    """Extract the magnitude or relevant parameters from a transformation YAML.
    Returns the value of 'magnitude' if present; otherwise returns a dict
    of all parameters excluding internal/ramp keys.  Returns None when the
    file does not exist or has no usable parameters.
    """
    if not yaml_path or not os.path.isfile(yaml_path):
        return None
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)
    params = data.get("parameters", {})
    if not params:
        return None
    if "magnitude" in params:
        return params["magnitude"]
    relevant = {k: v for k, v in params.items() if k not in SKIP_PARAMS}
    if not relevant:
        return None
    if len(relevant) == 1:
        val = list(relevant.values())[0]
        return val if val is not None else {}
    return relevant

def resolve_yaml(tx_code):
    """Resolve a TX code to its actual YAML file path via case-insensitive lookup."""
    key = tx_to_yaml_key(tx_code)
    actual = yaml_index.get(key)
    return os.path.join(transformations_dir, actual) if actual else None

# --- Step 6: For each transformation, get base + per-pathway parameters ---
pw_keys = list(pathways.keys())

for code, info in transformations.items():
    base_yaml = info["transformation_yaml_name"]
    base_path = os.path.join(transformations_dir, base_yaml)
    info["magnitude_base"] = extract_parameters(base_path)

    for pk in pw_keys:
        tx = pw_tx_map.get(pk, {}).get(code)
        if tx is None:
            info[f"magnitude_{pk}"] = None
        elif tx == code:
            info[f"magnitude_{pk}"] = info["magnitude_base"]
        else:
            pw_path = resolve_yaml(tx)
            if pw_path is None:
                print(f"  YAML not found: {tx_to_yaml_key(tx)} (from {tx})")
            info[f"magnitude_{pk}"] = extract_parameters(pw_path)

# --- Step 7: Build the output DataFrame ---
df = pd.DataFrame.from_dict(transformations, orient="index")
df.index.name = "transformation_code"
df = df.reset_index()
print(f"{len(df)} transformations loaded across {len(pw_keys)} pathways")
df

  YAML not found: transformation_scoe_shift_fuel_heat_strategy_ndc_25.yaml (from TX:SCOE:SHIFT_FUEL_HEAT_STRATEGY_NDC_25)
  YAML not found: transformation_scoe_shift_fuel_heat_strategy_ndc_25.yaml (from TX:SCOE:SHIFT_FUEL_HEAT_STRATEGY_NDC_25)
  YAML not found: transformation_lndu_dec_wetland_loss_ndc_25.yaml (from TX:LNDU:DEC_WETLAND_LOSS_NDC_25)
  YAML not found: transformation_lndu_dec_wetland_loss_nz.yaml (from TX:LNDU:DEC_WETLAND_LOSS_NZ)
68 transformations loaded across 5 pathways


,transformation_code,subsector,transformation_name,transformation_yaml_name,transformation_description,magnitude_base,magnitude_PFLO:NDC_25_SCOE_ELEC,magnitude_PFLO:NDC_25_UNCONDITIONAL,magnitude_PFLO:NDC_25_UNCONDITIONAL_CE,magnitude_PFLO:HBLE,magnitude_PFLO:BAU
0,TX:AGRC:DEC_CH4_RICE,AGRC,Improve rice management,transformation_agrc_dec_ch4_rice.yaml,Many practices can reduce emissions associated...,0.45,None,None,None,0.45,NaN
1,TX:AGRC:DEC_EXPORTS,AGRC,Decrease Exports,transformation_agrc_dec_exports.yaml,Decrease agricultural exports by some percenta...,0.5,None,None,None,None,NaN
2,TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN,AGRC,Reduce supply chain losses,transformation_agrc_dec_losses_supply_chain.yaml,Reduce waste food waste in the agricultural (c...,0.3,None,None,None,0.5,NaN
3,TX:AGRC:INC_CONSERVATION_AGRICULTURE,AGRC,Expand conservation agriculture,transformation_agrc_inc_conservation_agricultu...,Conservation agriculture is the term given to ...,"{'dict_categories_to_magnitude': None, 'magnit...","{'dict_categories_to_magnitude': None, 'magnit...",None,None,"{'dict_categories_to_magnitude': None, 'magnit...",NaN
4,TX:AGRC:INC_PRODUCTIVITY,AGRC,Improve crop productivity,transformation_agrc_inc_productivity.yaml,Apply a fractional increase to crop yield fact...,0.2,0.6,None,0.6,0.7,0.2
...,...,...,...,...,...,...,...,...,...,...,...
63,TX:LNDU:PLUR,LNDU,Plur,lndu_plur.yaml,,None,0.275,None,None,None,NaN
64,TX:SCOE:SHIFT_FUEL_HEAT_STRATEGY_NDC_25_ELEC_RES,SCOE,Shift Fuel Heat Strategy Ndc 25 Elec Res,scoe_shift_fuel_heat_strategy_ndc_25_elec_res....,,None,None,None,None,None,NaN
65,TX:SCOE:INC_EFFICIENCY_HEAT,SCOE,Inc Efficiency Heat,scoe_inc_efficiency_heat.yaml,,None,{'fuel_biomass': 0.33},{'fuel_biomass': 0.33},{'fuel_biomass': 0.33},{'fuel_biomass': 0.33},NaN
66,TX:FRST:INCREASE_SEQUESTRATION,FRST,Increase Sequestration,frst_increase_sequestration.yaml,,None,0.1,None,None,0.15,NaN


In [4]:
#df.to_csv("../transformations_description/transformations_description.csv", index=False)

In [5]:
#df = pd.read_csv("/Users/alexa/Projects/ssp_uganda_data/transformations_description/transformations_description.csv")

In [6]:
import json
from transformation_templates_v2 import PathwayConfig, build_table_rows

# --- Uganda pathway configuration ---
# Each tuple is (column_suffix, display_label).
# column_suffix must match strategy_code values used when building the DataFrame.
cfg = PathwayConfig(pathways=[
    ("PFLO:BAU", "BAU"),
    ("PFLO:NDC_25_SCOE_ELEC", "NDC2 Conditional"),
    ("PFLO:NDC_25_UNCONDITIONAL", "NDC2 Unconditional (As Planned)"),
    ("PFLO:NDC_25_UNCONDITIONAL_CE", "NDC2 Unconditional w/Additional Cost Effective"),
    ("PFLO:HBLE", "Candidate NDC3"),
])

rows = build_table_rows(df, config=cfg)
print(f"{len(rows)} transformations loaded")


68 transformations loaded


In [7]:
# --- Export annex_data.json for the Word table generator ---
output_json = "../transformations_description/annex_data.json"
with open(output_json, 'w') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print(f"Exported → {output_json}")

Exported → ../transformations_description/annex_data.json


In [8]:
# --- Generate the Word annex table ---
# Requires: node + docx npm package (npm install -g docx)
# generate_annex_table.js must be in the same folder as this notebook.
import subprocess
result = subprocess.run(
    ["node", "generate_annex_table.js"],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

Done → annex_transformations.docx



In [9]:
# --- Export an Excel version of the annex table ---
excel_rows = []
for r in rows:
    row_dict = {
        "Subsector": r["subsector_label"],
        "Transformation": r["transformation_name"],
        "Policy Description": r["policy_description"],
    }
    for pw in r["pathways"]:
        row_dict[pw["label"]] = pw["text"]
    excel_rows.append(row_dict)

df_annex = pd.DataFrame(excel_rows)
excel_path = "../transformations_description/annex_transformations_Uganda.xlsx"
df_annex.to_excel(excel_path, index=False, sheet_name="Annex")
print(f"Exported → {excel_path}")

Exported → ../transformations_description/annex_transformations_Uganda.xlsx
